# 06 -- Linear models (logistic regression)

Baseline PD model and scorecard backbone. Logistic regression is
interpretable and the natural reference every other family must beat.

This notebook:
1. Tunes regularisation strength, penalty, and class weighting with
   `GridSearchCV` over the saved stratified folds.
2. Shows validation and learning curves (bias/variance).
3. Plots an epoch-wise train/validation **loss curve** (log loss).
4. Evaluates the selected model once on the untouched holdout.

All preprocessing (impute/cap/missing-flags) is refit inside every CV
fold via `src.modeling.make_pipeline`, so there is no leakage.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    classification_metrics,
    cv_results_frame,
    evaluate_split,
    load_modeling_data,
    make_pipeline,
    metrics_frame,
    run_grid_search,
    subsample,
    train_valid_report,
)
from src.modeling import plots

data = load_modeling_data()
pd.Series(data.summary())


In [ ]:
# Fraction of the TRAIN partition used for GridSearchCV. The holdout is never
# touched during tuning. Lower this (e.g. 0.3) for faster iteration, then set
# back to 1.0 to confirm the final model on the full training data.
SEARCH_FRAC = 1.0
search = subsample(data, frac=SEARCH_FRAC)
pd.Series(search.summary())


## Hyperparameter search

Grid over `C`, `penalty`, and `class_weight`. AUC drives selection.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=2000, solver="liblinear")
param_grid = {
    "C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0],
    "penalty": ["l1", "l2"],
    "class_weight": [None, "balanced"],
}
gs = run_grid_search(logreg, param_grid, search, refit="roc_auc")
cv_results_frame(gs).head(10)

## Validation and learning curves

In [ ]:
from sklearn.linear_model import LogisticRegression

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plots.plot_validation_curve(
    make_pipeline(LogisticRegression(max_iter=2000, solver="liblinear", class_weight="balanced")),
    search.X_train, search.y_train,
    param_name="model__C",
    param_range=[0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
    cv=search.cv, semilog=True, ax=axes[0],
)
plots.plot_learning_curve(gs.best_estimator_, search.X_train, search.y_train, cv=search.cv, ax=axes[1])
plt.tight_layout()
plt.show()

## Loss curve (train vs validation)

Logistic regression has no native iteration history, so we fit an
equivalent log-loss SGD model incrementally and record log loss per epoch.

In [ ]:
from src.modeling.linear import fit_logreg_sgd_curve

lc = fit_logreg_sgd_curve(
    search.X_train, search.y_train, data.X_holdout, data.y_holdout, n_epochs=60
)
plots.plot_loss_curve(lc.train_loss, lc.valid_loss, title="Logistic regression (SGD) loss curve")
plt.show()

## Holdout evaluation

In [ ]:
# Train vs holdout metrics for the selected model (overfitting check)
report = train_valid_report(gs.best_estimator_, data, valid_as_holdout=True)
report.round(4)


In [ ]:
# Holdout diagnostics for the selected model
best = gs.best_estimator_
proba_ho = best.predict_proba(data.X_holdout)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
plots.plot_roc_pr(data.y_holdout.to_numpy(), proba_ho, label="logreg", axes=axes[0])
plots.plot_calibration(data.y_holdout.to_numpy(), proba_ho, ax=axes[1][0], label="logreg")
plots.plot_score_distribution(data.y_holdout.to_numpy(), proba_ho, ax=axes[1][1])
plt.tight_layout()
plt.show()

best_t = classification_metrics(data.y_holdout.to_numpy(), proba_ho)["best_f1_threshold"]
fig, ax = plt.subplots(figsize=(5, 5))
plots.plot_confusion(data.y_holdout.to_numpy(), proba_ho, threshold=best_t, ax=ax)
plt.show()


## Coefficients

Standardised-scale coefficients ranked by absolute weight.

In [ ]:
pre = gs.best_estimator_.named_steps["preprocess"]
feat_names = list(pre.transform(search.X_train.iloc[:1]).columns)
coef = pd.Series(gs.best_estimator_.named_steps["model"].coef_[0], index=feat_names)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(20).to_frame("coef")

## Takeaways

- Logistic regression gives the interpretable baseline AUC/KS on holdout.
- `class_weight="balanced"` compensates for the ~8% default rate.
- The gap between train and validation curves shows regularisation is
  controlling variance.
- Next families (trees, ensembles, boosting) are judged against this.